In [5]:
import os
from langchain.chat_models import init_chat_model


llm = init_chat_model(
    model="openai/gpt-oss-120b",
    model_provider="openai",
    api_key=os.getenv("NOVITA_API_KEY"),
    base_url="https://api.novita.ai/openai",
)
# llm = init_chat_model(model="ollama:qwen2.5:3b")

In [6]:
llm.invoke("hello")

AIMessage(content='Hello! 👋 How can I help you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 58, 'prompt_tokens': 55, 'total_tokens': 113, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': '', 'id': '4cc878e69a49ffb97ecf884b6075bb57', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019bff10-3e64-7f01-9958-2844fbdc0ed7-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 55, 'output_tokens': 58, 'total_tokens': 113, 'input_token_details': {}, 'output_token_details': {}})

### LLMs and Augmentations

Augmentation is a technique to integrate any of the following components to the LLM to create agent or workflow for any use case:

- Retrieval
- Tools
- Structured output
- Memory

These are extensions needed to make the system intelligent and customize based on the use case.
For example, if we want an order management ai workflow integrated with LLM, we need all of them, see below for why:
- Retrieval: This helps in finding products and related details
- Tools: Retrieval could be added in this, checkout creation in db, and related crud operations, user data, products data operations. Tools are extensions for the LLM.



In [7]:
# Schema for structured output
from pydantic import BaseModel, Field


class SearchQuery(BaseModel):
    search_query: str = Field(None, description="Query that is optimized web search.")
    justification: str = Field(
        None, description="Why this query is relevant to the user's request."
    )


# Augment the LLM with schema for structured output
structured_llm = llm.with_structured_output(SearchQuery)

# Invoke the augmented LLM
output = structured_llm.invoke("How does Calcium CT score relate to high cholesterol?")
output

SearchQuery(search_query='coronary artery calcium CT score relationship with high cholesterol LDL total cholesterol studies 2020..2024', justification='The user wants to understand the relationship between coronary artery calcium (Ca) CT scores and high cholesterol levels. A focused search for recent medical literature, guidelines, and studies discussing how CAC scores correlate with cholesterol, especially LDL and total cholesterol, will provide the needed information.')

In [9]:
# Define a tool
def multiply(a: int, b: int) -> int:
    return a * b

# Augment the LLM with tools
llm_with_tools = llm.bind_tools([multiply])

# Invoke the LLM with input that triggers the tool call
msg = llm_with_tools.invoke("What is 2 times 3?")

# Get the tool call
print(msg)
msg.tool_calls

content='The result is **6**.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 86, 'prompt_tokens': 130, 'total_tokens': 216, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': '', 'id': '1e660728c86ee903ff67f4c3098ab794', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--019bff10-7410-7ed0-b78c-9f71b82e57c9-0' tool_calls=[{'name': 'multiply', 'args': {'a': 2, 'b': 3}, 'id': 'call_d26a4684f950487b963efb8a', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 130, 'output_tokens': 86, 'total_tokens': 216, 'input_token_details': {}, 'output_token_details': {}}


[{'name': 'multiply',
  'args': {'a': 2, 'b': 3},
  'id': 'call_d26a4684f950487b963efb8a',
  'type': 'tool_call'}]